In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import re, os, time
import Md2TeX
import Md2Py

In [ ]:
Md2Py.convert("Table.md")
Md2TeX.convert('Table.md')

In [ ]:
from Table import PaperData, papers_list

# Configure style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

In [ ]:
# Initialize collector
papers = PaperData()

for paper in papers_list:
    papers.add_entry(paper)

# Convert to DataFrame when ready for analysis
research_df = papers.to_dataframe()

In [ ]:
print("DataFrame created with {} entries.".format(len(research_df)))

## Basic graphs

In [ ]:
# Extract Architecture and Flexibility
research_df[['Accelerator\'s Architecture', 'NN Flexibility']] = research_df['Accelerator\'s Architecture\n(NN Flexibility)'].str.extract(r'([^\(]+)(?:\(([^\)]+)\))?', expand=True)
# Clean up whitespace
research_df['Accelerator\'s Architecture'] = research_df['Accelerator\'s Architecture'].str.strip()
research_df['NN Flexibility'] = research_df['NN Flexibility'].str.strip()

# Remove '-' or '--' entries
research_df['Accelerator\'s Architecture'] = research_df['Accelerator\'s Architecture'].replace({'-': "", '--': ""})
research_df['NN Flexibility'] = research_df['NN Flexibility'].replace({'-': "", '--': ""})

In [ ]:
IMG_DIR = "../PhD_Survey_1/Images/"
# Function to save figures
def save_fig(fig, title):
    # Create directory if it doesn't exist
    if not os.path.exists("Images"):
        os.makedirs("Images")
    # Remove spaces and other weird characters from the title
    title = re.sub(r'[^a-zA-Z0-9]', '_', title)
    # Remove leading and trailing underscores
    title = title.strip('_')
    # Save the figure as an image
    fig.write_image(f"{IMG_DIR}/{title}.pdf") # dummy save to prevent "Loading [MathJax]/extensions/MathMenu.js" bug 
    time.sleep(2)
    fig.write_image(f"{IMG_DIR}/{title}.pdf", format="pdf", width=600, height=400)

# function to create histogram
def histogram_by_year(dataframe, sperator, title):
    year_order = ['2014','2017','2018','2019','2020','2021','2022','2023','2024','2025']
    fig = px.histogram(
        dataframe,
        x='Year',
        color=sperator,
        barmode='stack',
        #title=title,
        category_orders={"Year": year_order},
        color_discrete_sequence=px.colors.qualitative.Bold,
        template="plotly_white",
    )
    fig.update_traces(
        marker=dict(line=dict(color='#ffffff', width=0)),
        textfont=dict(size=14),
    )
    fig.update_layout(
        legend=dict(font=dict(size=14), x=1.02, xanchor="left"),
        margin=dict(t=30, b=40, l=50, r=200),
        xaxis=dict(title_font=dict(size=18), tickfont=dict(size=16)),
        yaxis=dict(title="Number of Papers", title_font=dict(size=18), tickfont=dict(size=16)),
    )
    fig.show()
    save_fig(fig, title)

tmp_df = research_df[['Year', 'Task\nType/s']].dropna()
tmp_df['Task\nType/s'] = tmp_df['Task\nType/s'].str.replace('\n---1---\n', ' or ')
histogram_by_year(tmp_df, 'Task\nType/s', 'Number of paper per year and their respective Network Type/s')

tmp_df = research_df[['Year', 'Accelerator\'s Architecture']].fillna("Unknown")
histogram_by_year(tmp_df, 'Accelerator\'s Architecture', 'Number of paper per year and their respective Accelerator\nArchitecture')


In [ ]:
# Function to print a Pie chart
# textinfo can be 'value', 'percent', 'label', or 'text'
def pie_chart(dataframe, column, title, textinfo='value'):
    """
    Create a pie chart from the specified column of the DataFrame.
    """
    fig = px.pie(
        dataframe, 
        names=column, 
        #title=title,
        hole=0,
        color_discrete_sequence=px.colors.qualitative.Bold,
        template="plotly_white",
    )
    fig.update_traces(
        marker=dict(line=dict(color='#ffffff', width=0)),
        textinfo=textinfo,
        textfont=dict(size=16),
    )
    fig.update_layout(
        legend_title_text=column,
        legend=dict(font=dict(size=16)),
        margin=dict(t=30, b=30, l=30, r=30),
    )
    fig.show()
    save_fig(fig, title)

In [ ]:
expanded_rows = []
for _, row in research_df.iterrows():
    flag = False
    if "Classification" in row['Task\nType/s']:
        # Create a new row for "Classification Network"
        row_classification = row.copy()
        row_classification['Task\nType/s'] = "Classification"
        expanded_rows.append(row_classification)
        flag = True
    if "Segmentation" in row['Task\nType/s']:
        # Create a new row for "Segmentation Network"
        row_segmentation = row.copy()
        row_segmentation['Task\nType/s'] = "Segmentation"
        expanded_rows.append(row_segmentation)
        flag = True
    if "Object Detection" in row['Task\nType/s']:
        # Create a new row for "Object Detection Network"
        row_detection = row.copy()
        row_detection['Task\nType/s'] = "Object Detection"
        expanded_rows.append(row_detection)
        flag = True
    if flag == False:
        # Keep the original row if no match
        expanded_rows.append(row)

tmp_df = pd.DataFrame(expanded_rows).reset_index(drop=True)

# Generate and display the pie chart using the pie_chart function
pie_chart(tmp_df, 'Task\nType/s', 'Distribution of NN Types per Paper')

In [ ]:
# Replace None values with "Unknown" for better visualization
tmp_df = research_df['Accelerator\'s Architecture'].fillna("Unknown")
#tmp_df['Accelerator\'s Architecture'] = tmp_df['Accelerator\'s Architecture'].str.replace('', 'Unknown')
pie_chart(tmp_df, 'Accelerator\'s Architecture', 'Distribution of Accelerator Architectures per Paper')

In [ ]:
# Replace None values with "Unknown" for better visualization
tmp_df = research_df['FPGA\nVendor'].fillna("Unknown")
pie_chart(tmp_df, 'FPGA\nVendor', 'Distribution of FPGA Vendors per Paper', textinfo='value')

In [ ]:
# Apply the function to the 'Hardware\nImplementation' column
tmp_df = research_df[['Year', 'Hardware\nImplementation']].fillna("Undisclosed")
tmp_df['Hardware<br>Implementation'] = tmp_df['Hardware\nImplementation'].str.replace('HLS', 'Vivado/Vitis HLS')
tmp_df['Hardware<br>Implementation'] = tmp_df['Hardware<br>Implementation'].str.replace('+ Vivado/Vitis', '+')
tmp_df['Hardware<br>Implementation'] = tmp_df['Hardware<br>Implementation'].str.replace('System', '')
tmp_df['Hardware<br>Implementation'] = tmp_df['Hardware<br>Implementation'].str.replace('Verilog HDL', '[System]Verilog')
tmp_df['Hardware<br>Implementation'] = tmp_df['Hardware<br>Implementation'].str.replace(' + HLS', '')
tmp_df['Hardware<br>Implementation'] = tmp_df['Hardware<br>Implementation'].str.replace('AMD DPU + Vitis AI', 'AMD DPU')
# Replace None values with "Unknown" for better visualization
tmp_df = tmp_df.fillna("Unknown")

pie_chart(tmp_df, 'Hardware<br>Implementation', 'Distribution of Hardware Implementation Methods per Paper')
histogram_by_year(tmp_df, 'Hardware<br>Implementation', 'Number of paper per year and their respective Hardware Implementation Methods')

In [ ]:
# Replace None values with "Unknown" for better visualization
tmp_df = research_df['FPGA\nVendor'].fillna("Unknown")
pie_chart(tmp_df, 'FPGA\nVendor', 'Distribution of FPGA Vendors per Paper')

In [ ]:
tmp_df = research_df["Training\nMethod"].dropna()
pie_chart(tmp_df, 'Training\nMethod', 'Distribution of Training Methods per Paper')

In [ ]:
tmp_df = research_df["Frequency\n(MHz)"].dropna()
tmp_df = tmp_df.astype(str).str.replace('-------', 'or')
tmp_df = tmp_df.astype(str).str.replace('{', '')
tmp_df = tmp_df.astype(str).str.replace('}', '')
pie_chart(tmp_df, 'Frequency\n(MHz)', 'Distribution of Frequencies per Paper')

## Deterministic analysis

In [ ]:
# Function that splits a row into two rows if a range is detected
def split_data_when_range(df):
    """
    Splits a DataFrame row into two rows if a range is detected in the specified column.
    """
    # Create a new DataFrame to store the split data
    is_split = False
    new_rows = []
    for _, row in df.iterrows():
        new_row = row.copy()
        for col in df.columns:
            # Skip processing if the column is not in the specified list
            if (col == 'Classification\n(%)') or (col == 'Frequency\n(MHz)'):
                continue
            # Check if the value is a string before attempting string operations
            if isinstance(row[col], str) and (' - ' in row[col]):
                text_row_1 = ""
                text_row_2 = ""
                if not is_split:
                    new_row_2 = row.copy()
                    is_split = True
                parts = row[col].split('\n')
                for part in parts:
                    # Check if the column contains a range
                    match = re.search(r'(\d+(?:\.\d+)?)\s+-\s+(\d+(?:\.\d+)?)\s+(\S*)', part)
                    if match:
                        text_row_1 += match.group(1).strip() + ' ' + match.group(3).strip() + '\n'
                        text_row_2 += match.group(2).strip() + ' ' + match.group(3).strip() + '\n'
                new_row[col] = text_row_1.strip()
                new_row_2[col] = text_row_2.strip()
                #print("Splitting row: ", row[col], " into ", new_row[col], " and ", new_row_2[col])
        new_rows.append(new_row)
        if is_split:
            new_rows.append(new_row_2)
            is_split = False

    new_df = pd.DataFrame(new_rows)
    return new_df

# Function to split data rows based on a specific string
def split_data_rows(df, split_string):
    # Create a new DataFrame to store the split data
    is_split = False
    new_rows = []
    for _, row in df.iterrows():
        new_rows_tmp = []
        for col in df.columns:
            if (not row[col] == None) and (split_string in row[col]):
                parts = row[col].split(split_string)
                for index, part in enumerate(parts):
                    if not is_split:
                        new_row = row.copy()
                        new_row[col] = part.strip()
                        new_rows_tmp.append(new_row)
                    else:
                        #print(row)
                        new_rows_tmp[index][col] = part.strip()
                is_split = True
        new_rows.extend(new_rows_tmp)
        if not is_split:
            new_rows.append(row)
        is_split = False

    new_df = pd.DataFrame(new_rows)
    return new_df

def pre_process_data(df):
    """
    Pre-process the DataFrame to prepare it for analysis.
    """
    # Replace commas in numeric strings
    for col in df.columns:
        if df[col].dtype == 'object':
            # Handle newlines within parentheses using DOTALL mode ((?s))
            df[col] = df[col].str.replace(r'(?s)\([^)]*\)', '', regex=True)
            df[col] = df[col].str.strip()
            # Clean up special characters that might cause parsing issues
            df[col] = df[col].str.replace(',', '')
            df[col] = df[col].str.replace('*', '')
            df[col] = df[col].str.replace('$', '')
            df[col] = df[col].str.replace('approx', '')
            df[col] = df[col].str.strip()
    
    # Look for all ------- and split data row in two rows, copying the common part
    df = split_data_rows(df, '-------')
    df = split_data_rows(df, '---1---')
    df = split_data_rows(df, '---2---')

    df = split_data_when_range(df)

    return df

In [ ]:
# Pre-process the DataFrame for analysis
df_for_analysis = pre_process_data(research_df)
# Display the first few rows of the cleaned DataFrame
print("DataFrame created with {} entries.".format(len(df_for_analysis)))

In [ ]:
print("DataFrame after pre-processing:")
df_for_analysis.to_csv('research_dataframe.csv')

### Plot functions

In [ ]:
# Function to print scatter plot
def fancy_scatter_plot(dataframe, x_col, y_col, title, hue=None, x_log=False, y_log=False, legend_x=1, legend_y=1, legend_xanchor='auto', legend_yanchor='auto'):
    """
    Create a scatter plot from the specified columns of the DataFrame.
    Calculates and displays a trendline if the 'trendline' argument is set (e.g., 'ols').
    """
    trendline = None
    #if not (x_log or y_log):
    #    trendline = "ols"
    trendline_options=dict(
        #log_x=x_log,
        log_y=y_log,
    )
    fig = px.scatter(
        dataframe,
        x=x_col,
        y=y_col,
        #title=title,
        color=hue,
        log_x=x_log,
        log_y=y_log,
        color_discrete_sequence=px.colors.qualitative.Bold,
        color_discrete_map={
            "Dataflow Architecture": px.colors.qualitative.Bold[0],
            "Time-Multiplexed Architecture": px.colors.qualitative.Bold[1]
        },
        category_orders={
            "Accelerator's Architecture": [
                "Dataflow Architecture",
                "Time-Multiplexed Architecture"
            ]
        },
        trendline=trendline,
        trendline_options=trendline_options,
        trendline_scope="overall",
        template="plotly_white",
    )
    fig.update_traces(
        marker=dict(line=dict(width=0)),
        textfont=dict(size=12),
    )
    legend_dict = dict(
        orientation="v",
        yanchor='auto',
        y=legend_y,
        xanchor='auto',
        x=legend_x,
        font=dict(size=14),
        bgcolor='rgba(255, 255, 255, 0.6)',
        bordercolor='rgba(0, 0, 0, 0.1)',
        borderwidth=1,
        traceorder='normal',
    )
    fig.update_layout(
        legend_title_text=hue,
        margin=dict(t=30, b=30, l=30, r=30),
        legend=legend_dict,
        xaxis=dict(title_font=dict(size=18), tickfont=dict(size=16)),
        yaxis=dict(title_font=dict(size=18), tickfont=dict(size=16)),
    )
    fig.show()
    save_fig(fig, title)


def scatter_plot(dataframe, x_col, y_col, title, hue=None, x_log=False, y_log=False, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='upper'):
    plt.figure()
    palette = {
        "Dataflow Architecture": sns.color_palette()[0],
        "Time-Multiplexed Architecture": sns.color_palette()[1]
    }
    sns.scatterplot(data=dataframe, x=x_col, y=y_col, hue=hue, palette=palette if hue == "Accelerator\'s Architecture" else None)
    if x_log:
        plt.xscale('log')
    if y_log:
        plt.yscale('log')
    plt.xlabel(x_col, fontsize=16)
    plt.ylabel(y_col, fontsize=16)
    plt.title(title, fontsize=16)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend(title=hue, bbox_to_anchor=(legend_x, legend_y), loc=f'{legend_yanchor} {legend_xanchor}')
    plt.show()

### Convert values functions

In [ ]:
# Function to convert energy units to Seconds
def convert_time_units(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(s|ms|mu s)', value)
        if match:
            num_value = float(match.group(1))
            unit = match.group(2)
            if unit == 'ms':
                return num_value / 1000  # Convert ms to s
            elif unit == 'mu s':
                return num_value / 1e6  # Convert us to s
            else:
                return num_value  # Already in Seconds
    return None

# Function to convert energy units to Joules
def convert_energy_units(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(J|mJ|mu J)', value)
        if match:
            num_value = float(match.group(1))
            unit = match.group(2)
            if unit == 'mJ':
                return num_value / 1000  # Convert mJ to J
            elif unit == 'mu J':
                return num_value / 1e6  # Convert uJ to J
            else:
                return num_value  # Already in Joules
    return None

# Function to convert Input Size to int
def convert_input_size(value):
    if isinstance(value, str):
        # Check if the string contains 'x', indicating dimensions
        try:
            parts = value.split('x')
            # Attempt to convert each part to an integer
            value = 1
            for part in parts:
                value = value * int(part)
            return value
        except ValueError:
            # If the string cannot be converted to an integer, return None
            return None
    return None

# Function to convert Number of Operations to OPs
def convert_ops(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(GOP|MOP|KOP|OP|GFLOP|MFLOP|KFLOP|FLOP)', value)
        if match:
            num_value = float(match.group(1))
            unit = match.group(2).upper()
            if (unit == 'GOP') or unit == 'GFLOP':
                return num_value * 1e9
            elif (unit == 'MOP') or unit == 'MFLOP':
                return num_value * 1e6
            elif (unit == 'KOP') or unit == 'KFLOP':
                return num_value * 1e3
            elif (unit == 'OP') or unit == 'FLOP':
                return num_value
    return None

# Function to convert Memory Size to Bytes
def convert_memory_size(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(B|KB|MB|GB|TB)', value)
        if match:
            num_value = float(match.group(1))
            unit = match.group(2).upper()
            if unit == 'KB':
                return num_value * 1024  # Convert KB to Bytes
            elif unit == 'MB':
                return num_value * 1024**2  # Convert MB to Bytes
            elif unit == 'GB':
                return num_value * 1024**3  # Convert GB to Bytes
            elif unit == 'TB':
                return num_value * 1024**4  # Convert TB to Bytes
            else:
                return num_value  # Already in Bytes
    return None

### Extract values functions

In [ ]:
# Function to extract DSPs and convert latency to milliseconds
def extract_dsp(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(DSP|DSPs)', value)
        if match:
            num_value = float(match.group(1))
            return num_value
    return None

# Function to extract BRAM
def extract_bram(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(BRAM|BRAMs)', value)
        if match:
            num_value = float(match.group(1))
            if "18k" in value:
                return num_value / 2
            return num_value
    return None

# Function to extract LUTs
def extract_lut(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(LUT|LUTs)', value)
        if match:
            num_value = float(match.group(1))
            return num_value
    return None

# Function to extract FF
def extract_ff(value):
    if isinstance(value, str):
        # Extract numeric value and unit
        match = re.search(r'(\d+(?:\.\d+)?)\s*(FF|FFs)', value)
        if match:
            num_value = float(match.group(1))
            return num_value
    return None

In [ ]:
# Apply the function to extract DSPs and latency
temp_df = df_for_analysis[['FPGA\nResources\nConsumption', 'Inference\nTime\n(Latency)', 'Accelerator\'s Architecture']].dropna()
temp_df['Inference Time (s)'] = temp_df['Inference\nTime\n(Latency)'].apply(convert_time_units)
temp_df['DSPs'] = temp_df['FPGA\nResources\nConsumption'].apply(extract_dsp)
temp_df = temp_df.dropna()
# Plot the data with a logarithmic scale on the y-axis and different colors for 'Accelerator\'s Architecture'
fancy_scatter_plot(temp_df, 'DSPs', 'Inference Time (s)', 'DSPs vs Inference Time (s)', hue='Accelerator\'s Architecture', x_log=False, y_log=True, legend_x=1, legend_y=0, legend_xanchor='right', legend_yanchor='bottom')


In [ ]:
temp_df = df_for_analysis[['FPGA\nResources\nConsumption', 'Performance\n(GOP/s)', 'Accelerator\'s Architecture']].dropna()
temp_df['BRAM'] = temp_df['FPGA\nResources\nConsumption'].apply(extract_bram)
temp_df['Performance\n(GOP/s)'] = temp_df['Performance\n(GOP/s)'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'BRAM', 'Performance\n(GOP/s)', 'BRAM vs Performance\n(GOP/s)', hue='Accelerator\'s Architecture', x_log=False, y_log=False, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='top')

In [ ]:
temp_df['DSPs'] = temp_df['FPGA\nResources\nConsumption'].apply(extract_dsp)
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'DSPs', 'Performance\n(GOP/s)', 'DSPs vs Performance\n(GOP/s)', hue='Accelerator\'s Architecture', x_log=False, y_log=False, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

In [ ]:
temp_df['LUT'] = temp_df['FPGA\nResources\nConsumption'].apply(extract_lut)
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'LUT', 'Performance\n(GOP/s)', 'LUTs vs Performance\n(GOP/s)', hue='Accelerator\'s Architecture', x_log=False, y_log=False, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

In [ ]:
temp_df['FF'] = temp_df['FPGA\nResources\nConsumption'].apply(extract_lut)
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'FF', 'Performance\n(GOP/s)', 'FFs vs Performance\n(GOP/s)', hue='Accelerator\'s Architecture', x_log=False, y_log=False, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

## Energy related graphs

In [ ]:
# Apply the function to extract BRAM and latency
temp_df = df_for_analysis[['Performance\n(GOP/s)', 'Power\nConsumption', 'Accelerator\'s Architecture']].dropna()

# Extract numeric values from 'Performance\n(GOP/s)' and convert to float
temp_df['Performance\n(GOP/s)'] = temp_df['Performance\n(GOP/s)'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
# Extract numeric values from 'Power\nConsumption' and convert to float
temp_df['Power Consumption (W)'] = temp_df['Power\nConsumption'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)

fancy_scatter_plot(temp_df, 'Performance\n(GOP/s)', 'Power Consumption (W)', 'Performance vs Power Consumption', hue='Accelerator\'s Architecture', x_log=False, y_log=False, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

In [ ]:
# Apply the function to extract BRAM and latency
temp_df = df_for_analysis[['Per-inference\nEnergy\nConsumption', 'No. of\nparameters', 'Accelerator\'s Architecture']].dropna()

# Extract numeric values from 'Performance\n(GOP/s)' and convert to float
temp_df['Per-inference\nEnergy\nConsumption'] = temp_df['Per-inference\nEnergy\nConsumption'].apply(convert_energy_units)
# Extract numeric values from 'Power\nConsumption' and convert to float
temp_df['No. of\nparameters'] = temp_df['No. of\nparameters'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'No. of\nparameters', 'Per-inference\nEnergy\nConsumption', 'Per-inference Energy Consumption vs Number of Parameters', hue='Accelerator\'s Architecture', x_log=True, y_log=True, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

In [ ]:
# Apply the function to extract BRAM and latency
temp_df = df_for_analysis[['Power\nConsumption', 'Frequency\n(MHz)', 'Accelerator\'s Architecture']].dropna()

# Extract numeric values from 'Power\nConsumption' and convert to float
temp_df['Power Consumption (W)'] = temp_df['Power\nConsumption'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
# Extract numeric values from 'Frequency\n(MHz)' and convert to float
temp_df['Frequency\n(MHz)'] = temp_df['Frequency\n(MHz)'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)


fancy_scatter_plot(temp_df, 'Frequency\n(MHz)', 'Power Consumption (W)', 'Power Consumption vs Frequency', hue='Accelerator\'s Architecture', x_log=False, y_log=False, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

In [ ]:
# Apply the function to extract DSPs and latency
temp_df = df_for_analysis[['FPGA\nResources\nConsumption', 'Per-inference\nEnergy\nConsumption', 'Accelerator\'s Architecture']].dropna()
temp_df['Per-inference\nEnergy\nConsumption'] = temp_df['Per-inference\nEnergy\nConsumption'].apply(convert_energy_units)
temp_df['DSPs'] = temp_df['FPGA\nResources\nConsumption'].apply(extract_dsp)
temp_df['BRAM'] = temp_df['FPGA\nResources\nConsumption'].apply(extract_bram)
# Remove rows with NaN values
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'BRAM', 'Per-inference\nEnergy\nConsumption', 'Number of BRAM vs Per-inference\nEnergy\nConsumption', hue='Accelerator\'s Architecture', x_log=False, y_log=True, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='top')

fancy_scatter_plot(temp_df, 'DSPs', 'Per-inference\nEnergy\nConsumption', 'Number of DSPs vs Per-inference\nEnergy\nConsumption', hue='Accelerator\'s Architecture', x_log=False, y_log=True, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='top')

In [ ]:
temp_df = df_for_analysis[['Operations\n(OP or FLOP)', 'Per-inference\nEnergy\nConsumption', 'No. of\nparameters', 'Throughput', 'Accelerator\'s Architecture']].dropna()

temp_df['Per-inference\nEnergy\nConsumption'] = temp_df['Per-inference\nEnergy\nConsumption'].apply(convert_energy_units)
temp_df['Operations\n(OP or FLOP)'] = temp_df['Operations\n(OP or FLOP)'].apply(convert_ops)
temp_df['No. of\nparameters'] = temp_df['No. of\nparameters'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
temp_df['Throughput'] = temp_df['Throughput'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
# Remove rows with NaN values
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'Operations\n(OP or FLOP)', 'Per-inference\nEnergy\nConsumption', 'Number of Operations vs Per-inference\nEnergy\nConsumption', hue='Accelerator\'s Architecture', x_log=True, y_log=True, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')
fancy_scatter_plot(temp_df, 'No. of\nparameters', 'Per-inference\nEnergy\nConsumption', 'Number of Parameters vs Per-inference\nEnergy\nConsumption', hue='Accelerator\'s Architecture', x_log=True, y_log=True, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

In [ ]:
temp_df = df_for_analysis[['Operations\n(OP or FLOP)', 'Inference\nTime\n(Latency)', 'No. of\nparameters', 'Throughput', 'Accelerator\'s Architecture']].dropna()

temp_df['Inference Time (s)'] = temp_df['Inference\nTime\n(Latency)'].apply(convert_time_units)
temp_df['Operations\n(OP or FLOP)'] = temp_df['Operations\n(OP or FLOP)'].apply(convert_ops)
temp_df['No. of\nparameters'] = temp_df['No. of\nparameters'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
temp_df['Throughput (FPS)'] = temp_df['Throughput'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
# Remove rows with NaN values
temp_df = temp_df.dropna()

fancy_scatter_plot(temp_df, 'Operations\n(OP or FLOP)', 'Throughput (FPS)', 'Number of Operations vs Throughput', hue='Accelerator\'s Architecture', x_log=True, y_log=True, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='top')
fancy_scatter_plot(temp_df, 'No. of\nparameters', 'Throughput (FPS)', 'Number of Parameters vs Throughput', hue='Accelerator\'s Architecture', x_log=False, y_log=True, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='top')
fancy_scatter_plot(temp_df, 'Throughput (FPS)', 'Operations\n(OP or FLOP)', 'Throughput vs Number of Operations', hue='Accelerator\'s Architecture', x_log=True, y_log=True, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='top')

In [ ]:
fancy_scatter_plot(temp_df, 'Operations\n(OP or FLOP)', 'Inference Time (s)', 'Number of Operations vs Inference Time', hue='Accelerator\'s Architecture', x_log=True, y_log=True, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')
fancy_scatter_plot(temp_df, 'No. of\nparameters', 'Inference Time (s)', 'Number of Parameters vs Inference Time', hue='Accelerator\'s Architecture', x_log=False, y_log=True, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

In [ ]:
temp_df = df_for_analysis[['Input Size', 'Per-inference\nEnergy\nConsumption', 'No. of\nparameters', 'Throughput', 'Accelerator\'s Architecture']].dropna()
temp_df['Input Size'] = temp_df['Input Size'].apply(convert_input_size)
temp_df['Throughput (FPS)'] = temp_df['Throughput'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)
temp_df['Per-inference\nEnergy\nConsumption'] = temp_df['Per-inference\nEnergy\nConsumption'].apply(convert_energy_units)
tmp_df = temp_df.dropna()

fancy_scatter_plot(tmp_df, 'Input Size', 'Per-inference\nEnergy\nConsumption', 'Input Size vs Per-inference\nEnergy\nConsumption', hue='Accelerator\'s Architecture', x_log=False, y_log=True, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')
fancy_scatter_plot(tmp_df, 'Input Size', 'Throughput (FPS)', 'Input Size vs Throughput', hue='Accelerator\'s Architecture', x_log=False, y_log=True, legend_x=1, legend_y=1, legend_xanchor='right', legend_yanchor='top')

In [ ]:
temp_df = df_for_analysis[['No. of\nparameters', 'Memory\nSize', 'Accelerator\'s Architecture']].dropna()
temp_df['Memory\nSize'] = temp_df['Memory\nSize'].apply(convert_memory_size)
temp_df['No. of\nparameters'] = temp_df['No. of\nparameters'].str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float)

fancy_scatter_plot(temp_df, 'No. of\nparameters', 'Memory\nSize', 'Number of Parameters vs Memory Size', hue='Accelerator\'s Architecture', x_log=False, y_log=False, legend_x=0, legend_y=1, legend_xanchor='left', legend_yanchor='top')

print(temp_df)

In [ ]:
# Data for the pie chart
labels = ["None"] * 36 + ["Mentioned"] * 7 + ["Mechanisms against SEEs"] * 3
df_pie = pd.DataFrame({'Radiation Consideration': labels})

# Generate and display the pie chart
pie_chart(df_pie, 'Radiation Consideration', 'Consideration of Single Event Effects (SEEs)', textinfo='value')
